In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (Hemonet)

This notebook curates the **HemoNet** dataset by integrating peptide sequences with different levels of annotation. The source provides explicitly hemolytic and non-hemolytic peptides, as well as additional clinical and external validation sequences without label information. Here we standardize all inputs, keep labeled and unlabeled data separate, perform duplicate consistency checks, and export curated datasets and metadata.

- **Toxic effect / endpoint:** hemolytic
- **Source:** hemonet
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads multiple FASTA sources** provided by HemoNet:
  - `hemolytic.fasta` → labeled as hemolytic (`label = 1`),
  - `non_hemolytic.fasta` → labeled as non-hemolytic (`label = 0`),
  - `clinical_data.fasta` → unlabeled (`label = 2`),
  - `external_validation_data.fasta` → unlabeled (`label = 2`).
- **Uses robust FASTA parsers** to handle files with non-standard formatting.
- **Builds two datasets**:
  - a **labeled dataset** (`label ∈ {0, 1}`) combining hemolytic and non-hemolytic peptides,
  - an **unlabeled dataset** (`label = 2`) containing clinical and external validation sequences.
- **Checks duplicated sequences** separately for labeled and unlabeled data:
  - unique sequences are retained,
  - duplicates with consistent labels are collapsed,
  - conflicting-label duplicates are flagged and exported as errors.
- **Builds metadata** from the project-wide Excel description sheet and appends QC statistics.
- **Exports curated outputs**:
  - `processed_hemolytic_dataset.csv` (labeled dataset),
  - `detected_unlabel_sequences.csv` (unlabeled dataset),
  - `detected_error_sequences.csv`,
  - `metadata.json`.

In [2]:
name_source = "hemonet"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_hemolytic = read_fasta_doc(f"{PATH_INPUT}/{name_source}/hemolytic.fasta")
df_hemolytic["label"] = 1

In [4]:
df_non_hemolytic = read_fasta_with_strange_character(f"{PATH_INPUT}/{name_source}/non_hemolytic.fasta")
df_non_hemolytic["label"] = 0

In [5]:
df_clinical = read_fasta_with_strange_character(f"{PATH_INPUT}/{name_source}/clinical_data.fasta")
df_clinical["label"] = 2 # There is no information about the labels of this source, therefore it will be identified with a 2

In [6]:
df_external = read_fasta_with_strange_character(f"{PATH_INPUT}/{name_source}/external_validation_data.fasta")
df_external["label"] = 2 # There is no information about the labels of this source, therefore it will be identified with a 2

- Concatenating dataset

In [7]:
df_hemonet = pd.concat(
    [df_hemolytic, df_non_hemolytic],
    ignore_index=True
)
df_hemonet = df_hemonet[["sequence", "label"]]
df_hemonet.shape

(5267, 2)

In [8]:
df_unlabel = pd.concat(
    [df_clinical, df_external],
    ignore_index=True
)
df_unlabel = df_unlabel[["sequence", "label"]]
df_unlabel.shape

(41, 2)

- Checking duplicates

In [9]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_hemonet, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [10]:
df_remove_duplicated_unlabel, df_errors_unlabel, df_unique_unlabel = processing_duplicated(df_unlabel, group_seq="sequence", sort_key="label")
df_full_unlabel = pd.concat([df_unique_unlabel, df_remove_duplicated_unlabel], axis=0)

- Working with metada

In [11]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [12]:
raw_total_sequences = (
    len(df_hemolytic)
    + len(df_non_hemolytic)
    + len(df_clinical)
    + len(df_external)
)

In [13]:
dict_metadata.update({
    "number_of_raw_sequences": int(raw_total_sequences),
    "number_of_sequences_retained": int(len(df_full) + len(df_full_unlabel)),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences" : len(df_errors),
    "number_of_unlabel_sequences" : len(df_full_unlabel),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2021,
 'last update date': datetime.datetime(2021, 3, 11, 0, 0),
 'download date': Timestamp('2024-08-01 00:00:00'),
 'file format': 'fasta;docx',
 'peptide property': 'hemolytic, toxic',
 'dataset information': 'No information;Positive;Positive, Negative;Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'No information',
 'repository or server': 'https://github.com/adibayaseen/HemoNet',
 'publication': 'https://www.worldscientific.com/doi/10.1142/S0219720021500219?rfr_dat=cr_pub++0pubmed&rfr_id=ori%3Arid%3Acrossref.org&url_ver=Z39.88-2003',
 'number_of_raw_sequences': 5308,
 'number_of_sequences_retained': 3308,
 'number_of_positive_sequences': 1422,
 'number_of_negative_sequences': 1851,
 'number_of_erroneous_sequences': 252,
 'number_of_unlabel_sequences': 35,
 'modified_sequences_included': False}

- Exporting data

In [14]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [15]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_hemolytic_dataset.csv", index=False)
df_full_unlabel.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_unlabel_sequences.csv", index=False)
df_errors.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/detected_error_sequences.csv", index=False)